# Project 43 — Explainable Disaster Severity Assessment
## Notebook 01: Data Pipeline
**C-DAC Mohali | ML to Generative AI & LLMs**  
Team: Anuksha | Rishika | Ipshita  
Dataset: AIDERv2 (16,723 aerial images, 4 classes)

## 1. Imports & environment check

In [ ]:
import os, sys, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

print("Python :", sys.version)
print("TF     :", tf.__version__)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU(s) :", [g.name for g in gpus])
else:
    print("No GPU — CPU only. Training will be slow; use B0 only.")


## 2. Install dependencies
Run this once, then restart the kernel if needed.

In [ ]:
# Run in terminal instead of notebook for cleaner output:
# pip install tensorflow tf-keras-vis fpdf2 opencv-python pillow scikit-learn matplotlib seaborn plotly streamlit

# Or uncomment below to install from notebook:
# import subprocess
# subprocess.run(["pip", "install", "-q",
#     "tf-keras-vis", "fpdf2", "opencv-python",
#     "pillow", "scikit-learn", "matplotlib", "seaborn", "plotly", "streamlit"])
print("Dependencies assumed installed. Run pip install if any import fails.")


## 3. Paths & config
The next cell auto-detects the project root by walking up from the current
working directory until it finds a folder containing `data/`. This works no
matter where Jupyter was launched from (project root, `notebooks/`, etc.).

In [ ]:
from pathlib import Path

def find_project_root(marker="data", max_up=5):
    """Walk up from cwd until a directory containing `marker/` is found."""
    cur = Path.cwd().resolve()
    for _ in range(max_up + 1):
        if (cur / marker).is_dir():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not locate project root containing '{marker}/' "
        f"starting from {Path.cwd()}"
    )

PROJECT_ROOT = find_project_root("data")
print(f"Project root resolved to: {PROJECT_ROOT}")

# ── Paths (anchored to project root, NOT cwd)
BASE      = str(PROJECT_ROOT / "data")
TRAIN_DIR = str(PROJECT_ROOT / "data" / "Train")
VAL_DIR   = str(PROJECT_ROOT / "data" / "Val")
TEST_DIR  = str(PROJECT_ROOT / "data" / "Test")
OUT_DIR   = str(PROJECT_ROOT / "outputs")

os.makedirs(OUT_DIR, exist_ok=True)

# ── Dataset config
CLASSES    = ['Earthquake', 'Fire', 'Flood', 'Normal']
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

# ── Verify folders exist
for split, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    status = 'OK' if os.path.isdir(path) else 'MISSING'
    print(f"{split:6s}: {path}  [{status}]")

## 4. Dataset structure inspection

In [ ]:
print("Folder structure:")
for split, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    print(f"\n{split}/")
    if os.path.isdir(path):
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                n = len(os.listdir(cls_path))
                print(f"  {cls}/  ({n} files)")
    else:
        print("  ← folder not found")


## 5. File extension check

In [ ]:
all_files = glob.glob(os.path.join(TRAIN_DIR, '*', '*'))
extensions = set(os.path.splitext(f)[1].lower() for f in all_files[:200])
print("File extensions found in Train:", extensions)

# Show 3 sample paths
print("\nSample file paths:")
for f in all_files[:3]:
    print(" ", f)


## 6. Class distribution

In [ ]:
def count_images(base_dir, classes):
    counts = {}
    for cls in classes:
        path = os.path.join(base_dir, cls)
        if os.path.isdir(path):
            counts[cls] = len([
                f for f in os.listdir(path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
        else:
            counts[cls] = 0
    return counts

splits = {'Train': TRAIN_DIR, 'Val': VAL_DIR, 'Test': TEST_DIR}
split_counts = {s: count_images(d, CLASSES) for s, d in splits.items()}

# Print table
print(f"{'Class':15s}", end='')
for s in splits: print(f"{s:>10s}", end='')
print()
print('-' * 45)
for cls in CLASSES:
    print(f"{cls:15s}", end='')
    for s in splits:
        print(f"{split_counts[s][cls]:>10d}", end='')
    print()
print('-' * 45)
print(f"{'TOTAL':15s}", end='')
for s in splits:
    print(f"{sum(split_counts[s].values()):>10d}", end='')
print()

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#E24B4A', '#EF9F27', '#378ADD', '#1D9E75']

for i, (split, counts) in enumerate(split_counts.items()):
    bars = axes[i].bar(counts.keys(), counts.values(), color=colors)
    axes[i].set_title(f'{split} set', fontsize=12)
    axes[i].set_ylabel('Image count')
    axes[i].tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, counts.values()):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     v + 20, str(v), ha='center', fontsize=9)

plt.suptitle('AIDERv2 — Class distribution across splits', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/class_distribution.png")


## 7. Class imbalance — compute weights
Earthquake has the fewest samples. Class weights compensate during training.

In [ ]:
train_counts = count_images(TRAIN_DIR, CLASSES)
total_train  = sum(train_counts.values())
num_classes  = len(CLASSES)

class_weight = {
    i: total_train / (num_classes * train_counts[cls])
    for i, cls in enumerate(CLASSES)
}

print("Class weights (higher = more underrepresented):")
for i, cls in enumerate(CLASSES):
    bar = '█' * int(class_weight[i] * 10)
    print(f"  {i} {cls:12s}: {class_weight[i]:.4f}  {bar}")

print("\nPass this dict to model.fit(class_weight=class_weight) in notebook 02.")


## 8. Sample images — one per class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, cls in enumerate(CLASSES):
    files = (glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')) +
             glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpeg')) +
             glob.glob(os.path.join(TRAIN_DIR, cls, '*.png')))
    if files:
        img = Image.open(files[0]).convert('RGB')
        axes[i].imshow(img)
        axes[i].set_title(f"{cls}\n{img.size[0]}x{img.size[1]}", fontsize=11)
    else:
        axes[i].set_title(f"{cls}\n(no images found)", fontsize=11, color='red')
    axes[i].axis('off')

plt.suptitle('AIDERv2 — Sample image per class', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/sample_images.png")


## 9. Image size verification
Confirm all images are 224×224 (or close). AIDERv2 is pre-resized but worth checking.

In [ ]:
sizes = []
for cls in CLASSES:
    files = glob.glob(os.path.join(TRAIN_DIR, cls, '*'))[:20]  # sample 20 per class
    for f in files:
        try:
            with Image.open(f) as img:
                sizes.append(img.size)
        except Exception:
            pass

unique_sizes = set(sizes)
print(f"Unique sizes found (sample of 80 images): {unique_sizes}")

if unique_sizes == {(224, 224)}:
    print("✓ All sampled images are 224×224 — no resizing needed.")
else:
    print("⚠ Mixed sizes detected — image_dataset_from_directory will resize automatically.")


## 10. tf.data pipeline — verify loading
This is a smoke test. The full pipeline with augmentation lives in notebook 02.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(images, labels):
    return tf.cast(images, tf.float32) / 255.0, labels

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
    class_names=CLASSES
).map(preprocess).prefetch(AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
    class_names=CLASSES
).map(preprocess).prefetch(AUTOTUNE)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
    class_names=CLASSES
).map(preprocess).prefetch(AUTOTUNE)

# NOTE: After .map(), the dataset object is a _PrefetchDataset and no longer
# carries the .class_names attribute. We use the CLASSES list (passed into
# image_dataset_from_directory above) as the authoritative class order.
print("Classes (label index : name):")
for i, name in enumerate(CLASSES):
    print(f"  {i}: {name}")
print("Train batches    :", len(train_ds))
print("Val batches      :", len(val_ds))
print("Test batches     :", len(test_ds))

# Grab one batch and check shape
for imgs, lbls in train_ds.take(1):
    print(f"\nBatch image shape : {imgs.shape}  (batch, H, W, C)")
    print(f"Batch label shape : {lbls.shape}  (batch, num_classes)")
    print(f"Pixel range       : [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]")

print("\nPipeline ready.")

## 11. Augmentation preview
Visualise what augmented images look like before training.

In [ ]:
from tensorflow.keras import layers

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
], name="augmentation")

# Take one batch, show original vs augmented for first 4 images
for imgs, lbls in train_ds.take(1):
    sample_imgs = imgs[:4]
    class_idx   = tf.argmax(lbls[:4], axis=1).numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i in range(4):
    orig = sample_imgs[i].numpy()
    aug  = augment(tf.expand_dims(sample_imgs[i], 0), training=True)[0].numpy()
    aug  = np.clip(aug, 0, 1)

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f"Original\n{CLASSES[class_idx[i]]}", fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(aug)
    axes[1, i].set_title("Augmented", fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Augmentation preview — top: original, bottom: augmented', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'augmentation_preview.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/augmentation_preview.png")


## 12. Pipeline summary
Everything notebook 02 needs is confirmed here.

In [ ]:
print("=" * 50)
print("PIPELINE SUMMARY")
print("=" * 50)
print(f"Dataset root  : data/")
print(f"Splits        : Train / Val / Test")
print(f"Classes       : {CLASSES}")
print(f"Class order   : alphabetical (TF default)")
print(f"Image size    : {IMG_SIZE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Normalisation : pixel / 255.0  → [0, 1]")
print(f"Augmentation  : flip, rotate±10%, zoom±10%, brightness±10%")
print()
print("Class weights for training:")
for i, cls in enumerate(CLASSES):
    print(f"  {i}: {cls:12s} → {class_weight[i]:.4f}")
print()
print("Outputs saved to outputs/:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f}")
print()
print("✓ Ready for 02_model_training.ipynb")
